# This part contains code that was removed from the Parallel Corpora section.
it was cool but hard to work with. This won't run if it's not in main.ipynb - Jack

---

From the excel sheets, we'll import them into pandas DataFrames for each of the languages.

In [ ]:
corpora_lang = ["english", "tagalog", "yami", "kapampangan", "pangasinense"]

eng_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="english-verses")
tgl_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="tagalog-verses")
tao_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="yami-verses")
pam_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="kapampangan-verses")
pag_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="pangasinense-verses")

dataframes = [ eng_df, tgl_df, tao_df, pam_df, pag_df ]

# rename verse cols https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html
eng_df = eng_df.rename(columns={'Verse': 'English'})
tgl_df = tgl_df.rename(columns={'Verse': 'Tagalog'})
tao_df = tao_df.rename(columns={'Verse': 'Yami'})
pam_df = pam_df.rename(columns={'Verse': 'Kapampangan'})
pag_df = pag_df.rename(columns={'Verse': 'Pangasinense'})

# handle dtypes
for i, df in enumerate(dataframes):
    for col in ['Book', 'Chapter #', 'Verse #', 'Verse']:
        if col in df.columns:
            dataframes[i][col] = df[col].astype(object)

for df in dataframes:
    print(df.info())

In [ ]:
# line counter
print(len(eng_df))
print(len(tgl_df))
print(len(tao_df))
print(len(pam_df))
print(len(pag_df))

After some digging, we found out that there are some lines that are really missing on the website. Some may have more lines than others as well due to translation. Here, we will rename the books so we can sort them properly later. Lines where there is no match in one language will contain a blank space.

In [ ]:
# mapper
def book_title_renamer(df):
    replacements = {
        "Mateo": "Matthew", "Matay": "Matthew",
        "Marcos": "Mark",   "Make": "Mark",
        "Lucas": "Luke",    "Locya": "Luke",
        "Juan": "John",     "Yowani": "John"
    }
    # if Mateo or Matay replace content to Matthew
    # if Marcos or Make replace content to Mark
    # if Lucas or Locya replace content to Luke
    # if Juan or Yowani replace content to Juan
    df["Book"] = df["Book"].replace(replacements)
    return df

tgl_df = book_title_renamer(tgl_df)
tao_df = book_title_renamer(tao_df)
pam_df = book_title_renamer(pam_df)
pag_df = book_title_renamer(pag_df)

In [ ]:
# english-tagalog (ENG-TGL)

eng_tgl = pd.merge(
    eng_df[['Book', 'Chapter #', 'Verse #', 'English']],
    tgl_df[['Book', 'Chapter #', 'Verse #', 'Tagalog']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
)

eng_tgl

Since the rest have ranged verse numbers, we'll just export all of the remaining dfs here. We'll save the existing one though to `parallel_corpora.xlsx`

Source: 

In [ ]:
with pd.ExcelWriter(output_excel_path, engine="openpyxl", mode="a", if_sheet_exists="new") as writer:
    try:
        #tgl_df.to_excel(writer, sheet_name="tgl_df", index=False)
        #tao_df.to_excel(writer, sheet_name="tao_df", index=False)
        #pam_df.to_excel(writer, sheet_name="pam_df", index=False)
        #pag_df.to_excel(writer, sheet_name="pag_df", index=False)
        eng_tgl.to_excel(writer, sheet_name="eng-tgl", index=False)
        print("Wrote sheet eng-tgl to", output_excel_path)
    except Exception as e:
        print(f"Error processing: {e}")

In [ ]:
'''# tagalog-pangasinense (TGL-PAG)
# pangasinense also has ranged verse numbers

tgl_pag = pd.merge(
    tgl_df[['Book', 'Chapter #', 'Verse #', 'Tagalog']],
    pag_df[['Book', 'Chapter #', 'Verse #', 'Pangasinense']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
).sort_values(['Book', 'Chapter #', 'Verse #'])
tgl_pag

# tagalog-kapampangan (TGL-PAM)
# kapampangan also has ranged verse numbers
 
tgl_pam = pd.merge(
    tgl_df[['Book', 'Chapter #', 'Verse #', 'Tagalog']],
    pam_df[['Book', 'Chapter #', 'Verse #', 'Kapampangan']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
).sort_values(['Book', 'Chapter #', 'Verse #'])
tgl_pam

# english-kapampangan (ENG-PAM)
# kapampangan also has ranged verse numbers
eng_pam = pd.merge(
    eng_df[['Book', 'Chapter #', 'Verse #', 'English']],
    pam_df[['Book', 'Chapter #', 'Verse #', 'Kapampangan']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
).sort_values(['Book', 'Chapter #', 'Verse #'])
eng_pam

# english-yami (ENG-TAO)
# yami also has ranged verse numbers
eng_tao = pd.merge(
    eng_df[['Book', 'Chapter #', 'Verse #', 'English']],
    tao_df[['Book', 'Chapter #', 'Verse #', 'Yami']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
).sort_values(['Book', 'Chapter #', 'Verse #'])
eng_tao

# tagalog-yami (TGL-TAO)
# yami also has ranged verse numbers
tgl_tao = pd.merge(
    tgl_df[['Book', 'Chapter #', 'Verse #', 'Tagalog']],
    tao_df[['Book', 'Chapter #', 'Verse #', 'Yami']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
).sort_values(['Book', 'Chapter #', 'Verse #'])
tgl_tao

# kapampangan-yami (PAM-TAO)
# kapampangan and yami have ranged verse numbers
pam_tao = pd.merge(
    pam_df[['Book', 'Chapter #', 'Verse #', 'Kapampangan']],
    tao_df[['Book', 'Chapter #', 'Verse #', 'Yami']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
).sort_values(['Book', 'Chapter #', 'Verse #'])
tgl_pag'''

Finally, we can turn them back into excel files or add them back to parallel_corpora.

Source: https://stackoverflow.com/questions/20219254/how-to-write-to-an-existing-excel-file-without-overwriting-data-using-pandas

In [ ]:
'''new_dfs = [eng_tgl, eng_tao, eng_pam, tgl_pam, tgl_tao, tgl_pag, pam_tao]

output_path = Path("parallel_corpora.xlsx")
for df in new_dfs:
    df.to_excel(output_path, index=False)'''